<h2 align="center" style="color:purple">Phase 2: AtliQ0 Bank Credit Card Project</h2>
    

### Business Analysis and launch of AB testing: Targeting Untapped Market

### Insights specific to customers with age group of 18 - 25
1. People with age group of 18 -25 accounts to ~25% of customer base in the data
2. Avg annual income of this age group is less than 50k
3. They don't have much credit history which is getting reflected in their credit score and max credit limit 
4. Usage of credit cards as payment type is relatively low compared to other groups
5. Avg transaction amount made with credit cards is also low compared to other groups
5. Top 3 most used shopping products categories  : Electronics, Fashion & Apparel, Beauty & Personal care

![Analysis Image](analysis.png)


In [40]:
## IMP libraries 
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
from scipy import stats as st
import seaborn as sns
import statsmodels.stats.api as sms
import statsmodels.api as sm

In [41]:
alpha = 0.05
power =0.8

## choosing sample size to launch the campagin

effects_sizes = [0.1,0.2,0.3,0.4,0.5,1]

for effect_size in effects_sizes :
    sample_size = sms.tt_ind_solve_power(
        effect_size=effect_size,
        alpha=alpha, 
        power=power,
        ratio= 1,
        alternative= 'two-sided'
    )
    print(f"Effect size: {effect_size}, Sample size: {int(sample_size)} customers")

Effect size: 0.1, Sample size: 1570 customers
Effect size: 0.2, Sample size: 393 customers
Effect size: 0.3, Sample size: 175 customers
Effect size: 0.4, Sample size: 99 customers
Effect size: 0.5, Sample size: 63 customers
Effect size: 1, Sample size: 16 customers


Based on business requirements, the test should be capable of detecting a minimum 0.4 standard deviation difference between the control and test groups. For the effect size 0.4, we need 100 customers and when we discussed with business, 100 customers is ok in terms of their budgeting constraints for this trail run

#### Forming control and test groups

1.We have identified approximately 246 customers within the age group of 18 to 25. From this pool, we will select 100 customers for the initial campaign launch.

2.The campaign is launched for 100 customers, as determined by the effective size calculation and by considering budgeting costs, and will run campaign for a duration of 2 months

3.Got a conversion rate of ~40% ( implies 40 out of 100 customers in test group started using credit card)

4.To maintain a similar sample size, a control group consisting of 40 customers will be created. Importantly, this control group will be completely exclusive of initial 100 customers used as test group.

5.So now we have 40 customers in each of control and test groups

At the end of the 2-month campaign period (from 09-10-23 to 11-10-23), we obtained daily data showing the average transaction amounts made by the entire group of 40 customers in both the control and test groups using existing and newly launched credit cards respectively
The key performance indicator (KPI) for this AB test aims to enhance average transaction amounts facilitated by the new card

In [12]:
## inserting data
df = pd.read_csv('/Users/darshanpatel/Desktop/chapter11_assets/data/avg_transactions_after_campaign.csv')

In [13]:
## cheching the data 
df.head(5)

,campaign_date,control_group_avg_tran,test_group_avg_tran
0,2023-09-10,259.83,277.32
1,2023-09-11,191.27,248.68
2,2023-09-12,212.41,286.61
3,2023-09-13,214.92,214.85
4,2023-09-14,158.55,344.08


In [14]:
df.tail(5)

,campaign_date,control_group_avg_tran,test_group_avg_tran
57,2023-11-06,255.70,140.61
58,2023-11-07,220.29,258.46
59,2023-11-08,204.72,249.35
60,2023-11-09,233.68,238.77
61,2023-11-10,206.80,221.60


In [21]:
df.shape

(62, 3)

In [20]:
## checking the Record where we had higer tranction then test group 

df[df['control_group_avg_tran'] > df['test_group_avg_tran']].shape 

(18, 3)

In [30]:
## Calculating the Percentage of Days Where Control Group Transactions Exceed the Test Group


p = df[df['control_group_avg_tran'] > df['test_group_avg_tran']].shape[0] / df.shape[0]
 
print(f' Percentage : {p * 100}')

 Percentage : 29.03225806451613


1.We will now use stats module from statmodels for doing Z-test

2.The order of passing control and test group data to sm.stats.ztest(test_data, control_data) defines the direction of the test and influences the test results.

3.When you pass test group data first, z-test module assumes that alternative hypothesis as mean of the test group is greater than the mean of the control group and conversely if you switch the order z-test module assumes alternative hypothesis as control group average is more than test group

4.In here we will be using order as sm.stats.ztest(test_group_data, control_group_data) based on our alternative hypothesis considered above.

5.By default z-test module in statmodels performs two tailed test. As we are doing one-tailed test in our case based on the direction and alternate hypothesis we have to set "alternative" parameter.

6.In out case based on test direction we will set "alternative" parameter to "larger"

How to choose right Alternative parameter
a.Two-tailed, meaning you are interested in identifying deviations across control and test groups in either direction

b.larger, This is a one-tailed test, specifically looking for whether the first group is significantly larger than the second

c.smaller, This is another one-tailed test, specifically looking for whether the first group is significantly smaller than the second

In [39]:
z_score , p_value = sm.stats.ztest(df['test_group_avg_tran'],df['control_group_avg_tran'],alternative = 'two-sided')
z_score , p_value

(2.7482973745691135, 0.005990564924405004)

In [32]:
z_critical = st.norm.ppf(1-alpha)
z_critical

1.6448536269514722

In [34]:
if z_score > z_critical:
    print("New credit card is more effective and has higher transactions.")
else :
    print('Old credit card is more effective and has higher transcations')

New credit card is more effective and has higher transactions.
